In [ ]:
import h5py
import os
import obspy
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- KONFIGURASI ---
BASE_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'
CSV_METADATA = os.path.join(BASE_DIR, 'Master_Metadata_Generated.csv')
OUTPUT_H5 = os.path.join(BASE_DIR, 'STEAD_Indonesia_Final_2004_2010.hdf5')

# 1. Load Metadata Master
print("Memuat Metadata Master...")
df_master = pd.read_csv(CSV_METADATA)
df_master.set_index('trace_name', inplace=True)

def pad_or_trim(data, n=6000):
    if len(data) >= n: return data[:n]
    else: return np.pad(data, (0, n - len(data)), mode='constant')

print("Memulai pembuatan Brankas HDF5 Final...")

# 2. Proses Merger
with h5py.File(OUTPUT_H5, 'w') as hf:
    for _, row in tqdm(df_master.iterrows(), total=len(df_master)):
        file_path = row['source_file']
        trace_name = row.name # Mengambil trace_name dari index
        
        try:
            st = obspy.read(file_path, headonly=True)
            if len(st) >= 3:
                # Membaca data penuh setelah validasi header
                st = obspy.read(file_path)
                data_list = [pad_or_trim(tr.data) for tr in st[:3]]
                data_array = np.column_stack(data_list)
                
                # Simpan dataset
                dset = hf.create_dataset(trace_name, data=data_array)
                
                # INJEKSI METADATA dari CSV Master ke dalam dset.attrs
                for col in df_master.columns:
                    val = row[col]
                    dset.attrs[col] = str(val) if pd.isna(val) else val
                    
        except Exception as e:
            continue

print(f"Selesai! Brankas HDF5 Final tersimpan di: {OUTPUT_H5}")

In [2]:
# -*- coding: utf-8 -*-
import os
import gc
import numpy as np
import pandas as pd
import h5py
from obspy import read
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ==============================================================================
# 🎛️ PARAMETER DIREKTORI
# ==============================================================================
OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv'
PATH_OUTPUT_HDF5 = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/hdf5_output/dataset_indonesia.hdf5'

INPUT_SIZE = 700  # 7 detik @ 100 Hz

# ==============================================================================
# 🔧 FUNGSI PRA-PEMROSESAN (SESUAI ARTIKEL MCU-QUAKE)
# ==============================================================================
def praproses_sesuai_artikel(arr):
    if len(arr) < INPUT_SIZE:
        arr = np.pad(arr, (0, INPUT_SIZE - len(arr)), 'constant')
    else:
        arr = arr[:INPUT_SIZE]
        
    max_val = np.max(np.abs(arr))
    if max_val > 0:
        arr = arr / max_val
        
    return arr.astype(np.float32)

# ==============================================================================
# 🧠 FUNGSI WORKER: MEMBACA DAN MEMBERSIHKAN DATA (EKSEKUSI PARALEL)
# ==============================================================================
def ekstrak_dan_bersihkan_mseed(item, dict_katalog):
    try:
        st = read(item['path'])
        st.detrend("demean")
        st.detrend("linear")
        st.resample(100.0)
        
        comp_z = st.select(component="Z")
        comp_n = st.select(component="N") or st.select(component="1")
        comp_e = st.select(component="E") or st.select(component="2")
        
        if not (comp_z and comp_n and comp_e): 
            return None
            
        # Potong tepat di Origin Time (Jendela 7 detik kaku untuk Dataset Benchmark)
        idx_start_le = int(60.0 * 100)
        idx_end_le = idx_start_le + INPUT_SIZE
        idx_start_no = int(10.0 * 100)
        idx_end_no = idx_start_no + INPUT_SIZE
        
        # Ekstraksi dan Normalisasi Gempa (LE)
        le_z = praproses_sesuai_artikel(comp_z[0].data[idx_start_le:idx_end_le])
        le_n = praproses_sesuai_artikel(comp_n[0].data[idx_start_le:idx_end_le])
        le_e = praproses_sesuai_artikel(comp_e[0].data[idx_start_le:idx_end_le])
        matriks_le_3c = np.vstack((le_z, le_n, le_e)) # Shape: (3, 700)
        
        # Ekstraksi dan Normalisasi Noise (NO)
        no_z = praproses_sesuai_artikel(comp_z[0].data[idx_start_no:idx_end_no])
        no_n = praproses_sesuai_artikel(comp_n[0].data[idx_start_no:idx_end_no])
        no_e = praproses_sesuai_artikel(comp_e[0].data[idx_start_no:idx_end_no])
        matriks_no_3c = np.vstack((no_z, no_n, no_e)) # Shape: (3, 700)
        
        # Ambil Metadata dari Kamus Katalog (O(1) Lookup)
        metadata = dict_katalog.get(item['eid'], {})
        
        return {
            'eid': item['eid'],
            'matriks_le': matriks_le_3c,
            'matriks_no': matriks_no_3c,
            'metadata': metadata
        }
    except Exception:
        return None

# ==============================================================================
# 🚀 CORE PIPELINE
# ==============================================================================
def jalankan_pipeline_hdf5():
    print("="*90)
    print("🚀 MEMULAI HDF5 DATA PIPELINE (ETL)")
    print("="*90)
    
    # 1. Muat Katalog dan Ubah ke Dictionary untuk Pencarian Super Cepat
    print("⏳ Memuat Katalog BMKG...")
    df_catalog = pd.read_csv(PATH_KATALOG_MASTER)
    df_catalog.columns = [col.lower() for col in df_catalog.columns]
    col_id = next((c for c in df_catalog.columns if 'id' in c), 'id')
    df_catalog[col_id] = df_catalog[col_id].astype(str)
    
    # Konversi ke dictionary agar worker tidak perlu melakukan df.loc berulang kali
    dict_katalog = df_catalog.set_index(col_id).to_dict(orient='index')
    
    # 2. Pindai File mseed
    daftar_file_mseed = []
    for root, _, files in os.walk(OUTPUT_WAVEFORM_DIR):
        for file in files:
            if file.endswith('.mseed'):
                parts = file.replace('.mseed', '').split('_')
                if len(parts) >= 3:
                    daftar_file_mseed.append({'path': os.path.join(root, file), 'eid': parts[2]})
                    
    print(f"   ✅ Ditemukan {len(daftar_file_mseed):,} file miniSEED siap diubah.")

    # 3. Proses Paralel (Baca & Bersihkan Data)
    hasil_ekstraksi = []
    MAX_THREADS = min(32, (os.cpu_count() or 1) + 4)
    print(f"\n⚡ Mengeksekusi Pra-pemrosesan dengan {MAX_THREADS} Workers...")
    
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = {executor.submit(ekstrak_dan_bersihkan_mseed, item, dict_katalog): item for item in daftar_file_mseed}
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Membaca mseed"):
            hasil = future.result()
            if hasil is not None:
                hasil_ekstraksi.append(hasil)

    # 4. Tulis ke HDF5 (Sekuensial agar file tidak korup)
    # 4. Tulis ke HDF5 (Sekuensial agar file tidak korup)
    print(f"\n📦 Menulis {len(hasil_ekstraksi):,} blok data ke dalam HDF5...")
    with h5py.File(PATH_OUTPUT_HDF5, 'w') as hdf:
        grp_le = hdf.create_group('earthquake')
        grp_no = hdf.create_group('noise')
        
        for data in tqdm(hasil_ekstraksi, desc="Menyusun HDF5"):
            eid = str(data['eid'])
            
            # 🔥 SOLUSI PENANGANAN DUPLIKAT: Pastikan nama dataset unik
            dataset_name = eid
            counter = 1
            while dataset_name in grp_le:
                dataset_name = f"{eid}_{counter}"
                counter += 1
            
            # Buat dataset matriks gelombang dengan nama yang dijamin unik
            dset_le = grp_le.create_dataset(dataset_name, data=data['matriks_le'], compression="gzip", compression_opts=4)
            dset_no = grp_no.create_dataset(dataset_name, data=data['matriks_no'], compression="gzip", compression_opts=4)
            
            # Tempelkan metadata BMKG sebagai Atribut (jika ada)
            for key, val in data['metadata'].items():
                # Pastikan format data didukung HDF5
                if pd.isna(val): val = "NaN"
                if isinstance(val, str): val = val.encode('utf-8') 
                
                dset_le.attrs[key] = val
                dset_no.attrs[key] = val
                
    print("\n" + "="*90)
    print(f"✅ PIPELINE SELESAI! File HDF5 berhasil dibentuk: \n📂 {PATH_OUTPUT_HDF5}")
    print("="*90)
    
    del hasil_ekstraksi, dict_katalog, df_catalog
    gc.collect()

if __name__ == "__main__":
    jalankan_pipeline_hdf5()

🚀 MEMULAI HDF5 DATA PIPELINE (ETL)
⏳ Memuat Katalog BMKG...
   ✅ Ditemukan 55,699 file miniSEED siap diubah.

⚡ Mengeksekusi Pra-pemrosesan dengan 15 Workers...


Membaca mseed: 100%|██████████| 55699/55699 [07:57<00:00, 116.67it/s]



📦 Menulis 55,699 blok data ke dalam HDF5...


Menyusun HDF5: 100%|██████████| 55699/55699 [00:30<00:00, 1842.86it/s]



✅ PIPELINE SELESAI! File HDF5 berhasil dibentuk: 
📂 /Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/hdf5_output/dataset_indonesia.hdf5
